# Hyperparameter Optimization with Amazon SageMaker XGBoost

This notebook will train a model which can be used to predict if a customer will enroll for a term deposit at a bank, after one or more phone calls. Hyperparameter tuning will be used in order to try multiple hyperparameter settings and produce the best model.

---

## Preparation

Let's start by specifying:

- The S3 bucket and prefix that you want to use for training and model data.  This should be within the same region as SageMaker training.
- The IAM role used to give training access to your data.

In [ ]:
import sagemaker
import boto3
from sagemaker.tuner import (
    IntegerParameter,
    ContinuousParameter,
    HyperparameterTuner,
)
from sagemaker.inputs import TrainingInput

import numpy as np
import pandas as pd
import os

region = boto3.Session().region_name
smclient = boto3.Session().client("sagemaker")

role = sagemaker.get_execution_role()

bucket = sagemaker.Session().default_bucket()
prefix = "sagemaker/4-HPO"

Now we'll copy the file to S3 for Amazon SageMaker training to pickup.

In [ ]:
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "train/train.csv")
).upload_file("./data/train.csv")
boto3.Session().resource("s3").Bucket(bucket).Object(
    os.path.join(prefix, "validation/validation.csv")
).upload_file("./data/validation.csv")

## Setup Hyperparameter Tuning 
*Note, with the default setting below, the hyperparameter tuning job can take about 30 minutes to complete.*

We will use SageMaker hyperparameter tuning to automate the searching process effectively. Specifically:
* We specify a range, or a list of possible values in the case of categorical hyperparameters, for each of the hyperparameter that we plan to tune.
* SageMaker hyperparameter tuning will:
    * Automatically launch multiple training jobs with different hyperparameter settings
    * Evaluate results of those training jobs based on a predefined "objective metric"
    * Select the hyperparameter settings for future attempts based on previous results. 
*For each hyperparameter tuning job, we will give it a budget (max number of training jobs) and it will complete once that many training jobs have been executed.

In this example, we are using SageMaker Python SDK to set up and manage the hyperparameter tuning job. We first configure the training jobs the hyperparameter tuning job will launch by initiating an estimator, which includes:
* The container image for the algorithm (XGBoost)
* Configuration for the output of the training jobs
* The values of static algorithm hyperparameters, those that are not specified will be given default values
* The type and number of instances to use for the training jobs

In [ ]:
sess = sagemaker.Session()

container = sagemaker.image_uris.retrieve("xgboost", boto3.Session().region_name, "3.0-5")

xgb = sagemaker.estimator.Estimator(
    container,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    output_path=f"s3://{bucket}/{prefix}/output",
    sagemaker_session=sess,
)

xgb.set_hyperparameters(
    eval_metric="auc",
    objective="binary:logistic",
    num_round=100,
    rate_drop=0.3,
    tweedie_variance_power=1.4,
)

We will tune four hyperparameters in this examples:

In [ ]:
hyperparameter_ranges = {
    "eta": ContinuousParameter(0, 1),
    "min_child_weight": ContinuousParameter(1, 10),
    "alpha": ContinuousParameter(0, 2),
    "max_depth": IntegerParameter(1, 10),
}

Next we'll specify the objective metric that we'd like to tune and its definition, which includes the regular expression (Regex) needed to extract that metric from the Cloudwatch logs of the training job.

Since we are using built-in XGBoost algorithm here, it emits two predefined metrics: *validation:auc* and *train:auc*, and we elected to monitor *validation:auc* as you can see below.

In this case, we only need to specify the metric name and do not need to provide regex. If you bring your own algorithm, your algorithm emits metrics by itself. In that case, you'll need to add a MetricDefinition object here to define the format of those metrics through regex, so that SageMaker knows how to extract those metrics from your CloudWatch logs.

In [ ]:
objective_metric_name = "validation:auc"

Now, we'll create a `HyperparameterTuner` object, to which we pass:
- The XGBoost estimator we created above
- Our hyperparameter ranges
- Objective metric name and definition
- Tuning resource configurations such as Number of training jobs to run in total and how many training jobs can be run in parallel.

In [ ]:
tuner = HyperparameterTuner(
    xgb, objective_metric_name, hyperparameter_ranges, max_jobs=20, max_parallel_jobs=3
)

## Launch Hyperparameter Tuning
Now we can launch a hyperparameter tuning job by calling *fit()* function. After the hyperparameter tuning job is created, we can go to SageMaker console to track the progress of the hyperparameter tuning job until it is completed.

In [ ]:
s3_input_train = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/train", content_type="csv"
)
s3_input_validation = TrainingInput(
    s3_data=f"s3://{bucket}/{prefix}/validation/".format(bucket, prefix), content_type="csv"
)

tuner.fit({"train": s3_input_train, "validation": s3_input_validation}, include_cls_metadata=False)

Let's just run a quick check of the hyperparameter tuning jobs status to make sure it started successfully.

In [ ]:
boto3.client("sagemaker").describe_hyper_parameter_tuning_job(
    HyperParameterTuningJobName=tuner.latest_tuning_job.job_name
)["HyperParameterTuningJobStatus"]

## Deploy the best model
Now that we have got the best model, we can deploy it to an endpoint.

We can get the best model from the hyperparameter tuning job and deploy it to an endpoint using the `deploy()` method. This will create a SageMaker hosted endpoint that we can use for inference.

In [ ]:
best_job_name = tuner.best_training_job()
print(f"Best job: {best_job_name}")

There is no need to locate the best training job, as the tuner directly deploys the best model.

In [ ]:
xgb_predictor = tuner.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
)

## Evaluation

Now we can evaluate the best model by making predictions on the test set. We configure the predictor to accept CSV input and output.

In [ ]:
from sagemaker.serializers import CSVSerializer

xgb_predictor.serializer = CSVSerializer()
xgb_predictor.content_type = "text/csv"

In [ ]:
def predict(data, predictor, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = ""
    for array in split_array:
        predictions = ",".join([predictions, predictor.predict(array).decode("utf-8")])
    return np.fromstring(predictions[1:], sep=",")

test_data = pd.read_csv("./data/test.csv", header=None)
predictions = predict(test_data.to_numpy()[:, 1:], xgb_predictor)

## Cleanup

When you're done, delete the endpoint to avoid ongoing charges.

In [ ]:
xgb_predictor.delete_endpoint(delete_endpoint_config=True)